# 04 — Mở rộng head và train các nhánh

**Input:** output NB1, NB2, NB3 (+ output version trước của chính NB4 nếu chạy tiếp). **GPU:** cần.
**Output:** `art/expand/`, `art/runs/<run>/`.

Mỗi session dừng trước khi vượt `kaggle.time_budget_h`. Chạy tiếp: Save Version → gắn output version vừa xong làm input → chạy lại;
run đã xong được copy sang output mới (output cuối cùng chứa đủ mọi run).

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"                                    # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C
cfg = C.load_config(SMOKE)
print(cfg["art"])

In [ ]:
STAGE = "train"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE)

In [ ]:
import hashlib, shutil, time
from pathlib import Path
from topicx import data as D, yolo as Y
from topicx.oracle import OracleStore
meta = C.load(C.find(cfg, "splits/meta.json"))
NOVEL, names = meta["novel"], meta["names"]
B = cfg["branches"] + cfg["analysis_branches"]
paths = D.split_paths(cfg)
pub_boxes = pd.read_csv(C.find(cfg, "splits/public_boxes.csv"))
replay = C.read_ids(C.find(cfg, "splits/replay_ids.txt"))
control = C.read_ids(C.find(cfg, "splits/control_ids.txt"))
dev = C.read_ids(C.find(cfg, "splits/dev_ids.txt"))
YD = C.SCRATCH / "yolo"
dev_full = D.write_yolo(YD / "dev_full", dev, pub_boxes, names, paths, cfg["img_wh"])
W = cfg["yolo"]["weights"]
base_w = C.find(cfg, f"base/run/{W}.pt")
base_meta = C.load(C.find(cfg, "base/meta.json"))
oracle = OracleStore(C.find(cfg, "splits/oracle"), C.out(cfg, "train/oracle_log.jsonl"))
ft, imgsz = cfg["yolo"]["finetune"], cfg["yolo"]["imgsz"]
est = lambda k: 1.3 * base_meta["sec_per_img_epoch"] * (len(replay) + k) * ft["epochs"]
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
deadline = time.time() + cfg["kaggle"]["time_budget_h"] * 3600

In [ ]:
def train_set(s, k, b):
    """(ids mới ngoài replay, boxes của chúng). Ảnh pool: nhãn đầy đủ base + novel từ oracle sau khi manifest đã chốt."""
    if b == "REPLAY_ONLY":
        return control[:k], pub_boxes[pub_boxes.image.isin(set(control[:k]))]
    man = C.find(cfg, f"selections/s{s}_k{k}_{b}.json")
    m = C.read_frozen(man)
    bx, _ = oracle.reveal(man)
    pad = m.get("pad_ids", [])
    return m["ids"] + pad, pd.concat([bx, pub_boxes[pub_boxes.image.isin(set(pad))]])

stopped = False
for s in cfg["seeds"]:
    exp = C.out(cfg, f"expand/s{s}/expanded.pt")
    prev = C.find(cfg, f"expand/s{s}/expanded.pt", required=False)
    if prev != exp and prev is not None:
        shutil.copytree(prev.parent, exp.parent, dirs_exist_ok=True)
    if not exp.exists():
        log = Y.expand_head(base_w, exp, names, seed=s)
        log["max_diff_vs_base"] = Y.check_expansion(base_w, exp)
        C.dump(log, exp.parent / "expand_log.json")
    for k in cfg["ks"]:
        for b in B:
            run = f"s{s}_k{k}_{b}"
            dst = C.WORK / cfg["art"] / "runs" / run
            prev = C.find(cfg, f"runs/{run}/result.json", required=False)
            if prev is not None:
                if prev.parent != dst:
                    shutil.copytree(prev.parent, dst, dirs_exist_ok=True)
                continue
            if stopped or time.time() + est(k) > deadline:
                stopped = True
                print("bỏ qua (hết ngân sách session):", run)
                continue
            new_ids, new_boxes = train_set(s, k, b)
            ids = replay + new_ids
            assert len(ids) == len(replay) + k and len(set(ids)) == len(ids), run
            boxes = pd.concat([pub_boxes[pub_boxes.image.isin(set(replay))], new_boxes])
            tr = D.write_yolo(YD / run, ids, boxes, names, paths, cfg["img_wh"])
            yml = D.write_data_yaml(YD / f"{run}.yaml", names, tr, dev_full)
            t = Y.train(exp, yml, dst, ft, seed=s, imgsz=imgsz)
            nb_ = new_boxes[new_boxes.image.isin(set(new_ids))]
            C.dump({"seed": s, "k": k, "branch": b, "n_images": len(ids), "n_new": len(new_ids),
                    "n_novel_instances": int((nb_.cls == NOVEL).sum()), "n_novel_images": int(nb_.image[nb_.cls == NOVEL].nunique()),
                    "n_base_instances_new": int((nb_.cls != NOVEL).sum()), "expanded_md5": md5(exp),
                    "replay_sha": C.ids_sha(replay), **t}, dst / "result.json")
            shutil.rmtree(C.SCRATCH / "runs" / run, ignore_errors=True)
            shutil.rmtree(YD / run, ignore_errors=True)
print("CÒN RUN CHƯA CHẠY -> Save Version, gắn output này làm input, chạy lại" if stopped else "đủ mọi run")